# Notebook 22.2 — Coder les bandits de zéro

**Chapitre 22 — Les problèmes de bandits (le RL dans sa forme la plus pure)**

Version notebook exécutable du fichier `22.3-bandits.py`. Reproduit le **10-armed testbed de Sutton & Barto (Chapitre 2)** et compare **6 agents** :

1. ε-greedy (ε=0.1)
2. ε-greedy (ε=0.01)
3. Optimistic Initial Values (Q₀=5)
4. UCB (c=2)
5. Thompson Sampling (gaussien)
6. Gradient Bandit (α=0.1)

**Vous allez :**
1. Implémenter l'environnement et les 6 agents.
2. Lancer l'expérience (récompense moyenne + % d'action optimale).
3. Comparer les courbes et le tableau de performances.
4. Modifier les hyperparamètres (ε, c, α, Q₀) et observer l'impact.

> Autonome : seuls `numpy` et `matplotlib` (déjà sur Colab) sont requis.

In [ ]:
from __future__ import annotations
import numpy as np
import matplotlib.pyplot as plt
print("Imports OK")

## 1. Environnement de bandit multi-bras

Les vraies valeurs `q*(a)` sont tirées de N(0,1). À chaque `step(a)`, la récompense est tirée de N(q*(a), 1).

In [ ]:
class BanditEnvironment:
    def __init__(self, k=10, seed=None):
        self.k = k
        self.rng = np.random.default_rng(seed)
        self.q_star = self.rng.normal(0.0, 1.0, size=k)
        self.optimal_action = int(np.argmax(self.q_star))

    def step(self, action):
        return float(self.rng.normal(self.q_star[action], 1.0))

env = BanditEnvironment(k=10, seed=0)
print("q*(a) =", np.round(env.q_star, 2))
print("Action optimale :", env.optimal_action)

## 2. Les 6 agents

In [ ]:
class EpsilonGreedyAgent:
    """epsilon-greedy avec moyenne empirique."""
    def __init__(self, k, epsilon=0.1, seed=None):
        self.k, self.epsilon = k, epsilon
        self.Q = np.zeros(k); self.N = np.zeros(k, dtype=int)
        self.rng = np.random.default_rng(seed)
    def select_action(self):
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(0, self.k))
        return int(np.argmax(self.Q))
    def update(self, a, r):
        self.N[a] += 1
        self.Q[a] += (r - self.Q[a]) / self.N[a]


class OptimisticAgent:
    """Q initialisé à une valeur optimiste -> exploration sans randomisation."""
    def __init__(self, k, q_init=5.0, alpha=0.1):
        self.k, self.alpha = k, alpha
        self.Q = np.full(k, q_init, dtype=float)
    def select_action(self):
        return int(np.argmax(self.Q))
    def update(self, a, r):
        self.Q[a] += self.alpha * (r - self.Q[a])


class UCBAgent:
    """Upper Confidence Bound : bonus d'incertitude c*sqrt(ln t / N(a))."""
    def __init__(self, k, c=2.0):
        self.k, self.c = k, c
        self.Q = np.zeros(k); self.N = np.zeros(k, dtype=int); self.t = 0
    def select_action(self):
        self.t += 1
        untried = np.where(self.N == 0)[0]
        if len(untried) > 0:
            return int(untried[0])
        bonus = self.c * np.sqrt(np.log(self.t) / self.N)
        return int(np.argmax(self.Q + bonus))
    def update(self, a, r):
        self.N[a] += 1
        self.Q[a] += (r - self.Q[a]) / self.N[a]


class ThompsonSamplingAgent:
    """Thompson Sampling bayésien (a priori gaussien)."""
    def __init__(self, k, sigma=1.0, seed=None):
        self.k, self.sigma = k, sigma
        self.mu = np.zeros(k); self.tau = np.full(k, 1.0 / sigma**2)
        self.rng = np.random.default_rng(seed)
    def select_action(self):
        sampled = self.rng.normal(self.mu, 1.0 / np.sqrt(self.tau))
        return int(np.argmax(sampled))
    def update(self, a, r):
        lik = 1.0 / self.sigma**2
        new_tau = self.tau[a] + lik
        self.mu[a] = (self.tau[a] * self.mu[a] + lik * r) / new_tau
        self.tau[a] = new_tau


class GradientBanditAgent:
    """Apprend des préférences H(a) via softmax + ascension de gradient (précurseur de REINFORCE)."""
    def __init__(self, k, alpha=0.1, seed=None):
        self.k, self.alpha = k, alpha
        self.H = np.zeros(k); self.avg_reward = 0.0; self.t = 0
        self.rng = np.random.default_rng(seed)
    def _softmax(self):
        e = np.exp(self.H - np.max(self.H))
        return e / np.sum(e)
    def select_action(self):
        return int(self.rng.choice(self.k, p=self._softmax()))
    def update(self, a, r):
        self.t += 1
        self.avg_reward += (r - self.avg_reward) / self.t
        probs = self._softmax()
        baseline = self.avg_reward
        for i in range(self.k):
            if i == a:
                self.H[i] += self.alpha * (r - baseline) * (1 - probs[i])
            else:
                self.H[i] -= self.alpha * (r - baseline) * probs[i]

print("6 agents définis.")

## 3. Boucle d'expérience

On moyenne sur plusieurs runs indépendants. Astuce : baissez `n_runs` (ex. 200) pour aller plus vite, augmentez-le pour des courbes plus lisses.

In [ ]:
def run_experiment(agent_factory, n_runs=200, n_steps=1000, k=10):
    rewards = np.zeros(n_steps)
    optimal_pct = np.zeros(n_steps)
    for run in range(n_runs):
        env = BanditEnvironment(k=k, seed=run)
        agent = agent_factory(k)
        for t in range(n_steps):
            a = agent.select_action()
            r = env.step(a)
            agent.update(a, r)
            rewards[t] += r
            if a == env.optimal_action:
                optimal_pct[t] += 1.0
    rewards /= n_runs
    optimal_pct = 100.0 * optimal_pct / n_runs
    return rewards, optimal_pct

agents = {
    "eps-greedy (0.1)":  lambda k: EpsilonGreedyAgent(k, epsilon=0.1, seed=0),
    "eps-greedy (0.01)": lambda k: EpsilonGreedyAgent(k, epsilon=0.01, seed=0),
    "Optimistic (Q0=5)": lambda k: OptimisticAgent(k, q_init=5.0, alpha=0.1),
    "UCB (c=2)":         lambda k: UCBAgent(k, c=2.0),
    "Thompson Sampling": lambda k: ThompsonSamplingAgent(k, seed=0),
    "Gradient (a=0.1)":  lambda k: GradientBanditAgent(k, alpha=0.1, seed=0),
}

N_RUNS, N_STEPS, K = 200, 1000, 10
print(f"Comparaison de {len(agents)} agents ({N_RUNS} runs x {N_STEPS} pas)...\n")
results = {}
for name, factory in agents.items():
    print(f"  Exécution de {name}...")
    results[name] = run_experiment(factory, n_runs=N_RUNS, n_steps=N_STEPS, k=K)
print("\nTerminé.")

## 4. Graphiques de comparaison

In [ ]:
colors = {
    "eps-greedy (0.1)": "#2563eb", "eps-greedy (0.01)": "#0ea5e9",
    "Optimistic (Q0=5)": "#f59e0b", "UCB (c=2)": "#9333ea",
    "Thompson Sampling": "#16a34a", "Gradient (a=0.1)": "#dc2626",
}

fig, axes = plt.subplots(2, 1, figsize=(12, 9))
for name, (rewards, _) in results.items():
    axes[0].plot(rewards, label=name, color=colors.get(name), linewidth=1.5)
axes[0].set_xlabel("Pas de temps"); axes[0].set_ylabel("Récompense moyenne")
axes[0].set_title("Bandits — Récompense moyenne (10-armed testbed)")
axes[0].legend(loc="lower right"); axes[0].grid(True, alpha=0.3)

for name, (_, optimal_pct) in results.items():
    axes[1].plot(optimal_pct, label=name, color=colors.get(name), linewidth=1.5)
axes[1].set_xlabel("Pas de temps"); axes[1].set_ylabel("% d'actions optimales")
axes[1].set_title("Bandits — Pourcentage d'action optimale")
axes[1].legend(loc="lower right"); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 5. Tableau des performances finales (moyenne des 100 derniers pas)

In [ ]:
print("-" * 60)
print(f"  {'Agent':<22} {'Récompense':>12} {'% Optimal':>12}")
print("-" * 60)
for name, (rewards, optimal_pct) in results.items():
    print(f"  {name:<22} {np.mean(rewards[-100:]):>12.3f} {np.mean(optimal_pct[-100:]):>11.1f}%")
print("-" * 60)

## 6. À vous de jouer

1. Faites varier `epsilon` (0.0, 0.01, 0.1, 0.3) : quel compromis exploration/exploitation ?
2. Changez `c` dans UCB (0.5, 1, 2, 4) : impact sur la vitesse de convergence ?
3. Testez `q_init` (0, 1, 5, 10) pour l'agent optimiste.
4. Rendez l'environnement **non-stationnaire** (faites dériver `q_star` par un petit random walk à chaque pas) : quel agent résiste le mieux ? (Indice : alpha constant.)
5. **Question finale :** classez les 6 algorithmes par regret final et reliez le résultat à la borne logarithmique de Lai & Robbins (sections 4 et 6 du chapitre).